In [15]:
# ============================================================
# H6: HYPERNETWORK + CLASS-BALANCED EPISODIC META-TRAINING
# ============================================================
#
# Key idea:
#
# META-TRAINING:
#   Student
#      ↓
#   class-balanced support/query episode
#      ↓
#   HyperNetwork personalization
#      ↓
#   standard cross-entropy
#
# TEST:
#   unseen student
#      ↓
#   first K real clips (EARLY)
#      OR
#   K real random clips (RANDOM)
#      ↓
#   personalization
#      ↓
#   remaining clips
#
# IMPORTANT:
# We DO NOT balance the test support set.
# The test protocol remains unchanged.
#
# H6 therefore tests whether minority-aware EPISODE
# CONSTRUCTION is more effective than simply weighting
# the loss.
# ============================================================

import argparse
import json
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIGURATION
# ============================================================

LABELS = [
    "Highly Disengage",
    "Disengage",
    "Engage",
    "Highly Engage",
]

LABEL_TO_ID = {
    label: i
    for i, label in enumerate(LABELS)
}

FEATURE_COLS = [
    f"emb_{i}"
    for i in range(1024)
]

NUM_CLASSES = 4
INPUT_DIM = 1024

EMBED_DIM = 128
CONTEXT_DIM = 128
HYPER_HIDDEN = 128

HEAD_PARAM_DIM = (
    NUM_CLASSES * EMBED_DIM
    + NUM_CLASSES
)

DEFAULT_EPISODES = 1000
DEFAULT_K_VALUES = [5, 10, 20]
DEFAULT_SEED = 42

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# DATA LOADING
# ============================================================

def parse_group_from_clip_id(clip_id: str):

    match = re.match(
        r"^(video[^_]+)_(\d+)_((?:person).+)$",
        str(clip_id),
    )

    if not match:
        return None

    return (
        f"{match.group(1)}_"
        f"{match.group(3)}"
    )


def load_clip_dataframe(json_path):

    with open(
        json_path,
        "r",
        encoding="utf-8",
    ) as f:

        data = json.load(f)

    records = []

    for clip_id, meta in data.items():

        if not isinstance(meta, dict):
            continue

        label = meta.get("label")

        if label not in LABELS:
            continue

        embeds = meta.get("embeds")

        if not isinstance(embeds, list):
            continue

        if len(embeds) != INPUT_DIM:
            continue

        group = parse_group_from_clip_id(
            clip_id
        )

        if group is None:
            continue

        arr = np.asarray(
            embeds,
            dtype=np.float32,
        )

        if not np.all(
            np.isfinite(arr)
        ):
            continue

        clip_index_match = re.search(
            r"_(\d+)_",
            str(clip_id),
        )

        clip_index = (
            int(
                clip_index_match.group(1)
            )
            if clip_index_match
            else 0
        )

        records.append(
            {
                "clip_id": str(clip_id),
                "group": group,
                "label": str(label),
                "label_id": LABEL_TO_ID[
                    str(label)
                ],
                "clip_index": clip_index,
                "embedding": arr,
            }
        )

    if not records:

        raise ValueError(
            f"No valid labelled clips "
            f"found in {json_path}"
        )

    df = pd.DataFrame(records)

    emb_matrix = np.vstack(
        df["embedding"].to_numpy()
    )

    emb_df = pd.DataFrame(
        emb_matrix,
        columns=FEATURE_COLS,
        index=df.index,
    )

    df = pd.concat(
        [
            df.drop(
                columns=["embedding"]
            ),
            emb_df,
        ],
        axis=1,
    )

    return df


def load_group_fold_map(
    fold_csv_path
):

    fold_df = pd.read_csv(
        fold_csv_path
    )

    required = {
        "group",
        "fold",
    }

    missing = (
        required
        - set(fold_df.columns)
    )

    if missing:

        raise ValueError(
            "Fold CSV missing required "
            f"columns: {sorted(missing)}"
        )

    group_fold = (
        fold_df[
            ["group", "fold"]
        ]
        .drop_duplicates()
    )

    check = (
        group_fold
        .groupby("group")["fold"]
        .nunique()
    )

    bad = check[
        check != 1
    ]

    if len(bad) > 0:

        raise ValueError(
            "Groups assigned to multiple "
            f"folds: {bad.index.tolist()}"
        )

    return dict(
        zip(
            group_fold["group"],
            group_fold["fold"].astype(int),
        )
    )


# ============================================================
# TEST SUPPORT / QUERY SAMPLING
# ============================================================

def sample_support_query(
    student_df,
    k_shot,
    protocol,
    rng,
):
    """
    TEST-TIME sampling.

    This function is deliberately unchanged from
    the original HyperNetwork experiment.

    EARLY:
        first K chronological clips.

    RANDOM:
        K randomly selected real clips.

    We DO NOT balance the test support set.
    """

    ordered = (
        student_df
        .sort_values("clip_index")
        .copy()
    )

    if len(ordered) <= k_shot:

        return (
            None,
            None,
            f"insufficient_clips_n="
            f"{len(ordered)}_k={k_shot}",
        )

    if protocol == "early":

        support = ordered.iloc[
            :k_shot
        ].copy()

    elif protocol == "random":

        sampled_idx = rng.choice(
            ordered.index.to_numpy(),
            size=k_shot,
            replace=False,
        )

        support = ordered.loc[
            sampled_idx
        ].copy()

    else:

        return (
            None,
            None,
            f"unknown_protocol_{protocol}",
        )

    query = ordered.loc[
        ~ordered.index.isin(
            support.index
        )
    ].copy()

    if query.empty:

        return (
            None,
            None,
            "empty_query",
        )

    return (
        support,
        query,
        None,
    )


# ============================================================
# H6 CLASS-BALANCED EPISODE SAMPLING
# ============================================================

def allocate_balanced_counts(
    available_counts,
    total_n,
):
    """
    Allocate approximately equal numbers of samples
    across classes that are actually available.

    Example:

        available:
            class 0 = 2
            class 1 = 100
            class 2 = 500
            class 3 = 20

        total_n = 20

    We try to obtain:

            2, ~6, ~6, ~6

    rather than:

            mostly class 2.

    No synthetic examples are created.

    If a class has fewer samples than requested,
    the remaining quota is redistributed.
    """

    available = {
        int(k): int(v)
        for k, v in available_counts.items()
        if int(v) > 0
    }

    if not available:
        return {}

    allocation = {
        cls: 0
        for cls in available
    }

    remaining = int(total_n)

    active = set(
        available.keys()
    )

    while (
        remaining > 0
        and active
    ):

        # Equal-share target among currently
        # active classes.
        share = max(
            1,
            remaining // len(active),
        )

        removed = set()

        for cls in sorted(active):

            capacity = (
                available[cls]
                - allocation[cls]
            )

            if capacity <= 0:

                removed.add(cls)
                continue

            amount = min(
                share,
                capacity,
                remaining,
            )

            allocation[cls] += amount
            remaining -= amount

            if (
                allocation[cls]
                >= available[cls]
            ):
                removed.add(cls)

            if remaining <= 0:
                break

        active -= removed

        # Safety fallback.
        # This should rarely be necessary.
        if (
            remaining > 0
            and not removed
        ):

            for cls in sorted(
                active
            ):

                capacity = (
                    available[cls]
                    - allocation[cls]
                )

                if capacity <= 0:
                    continue

                allocation[cls] += 1
                remaining -= 1

                if remaining <= 0:
                    break

    return allocation


def sample_balanced_subset(
    student_df,
    n_samples,
    rng,
):
    """
    Select n_samples from a student's clips while
    making the class distribution as balanced as
    possible.

    Only real clips are selected.

    Classes absent from the student are ignored.
    """

    if len(student_df) < n_samples:

        return None

    class_counts = (
        student_df[
            "label_id"
        ]
        .value_counts()
        .to_dict()
    )

    allocation = allocate_balanced_counts(
        class_counts,
        n_samples,
    )

    selected_indices = []

    for class_id, amount in allocation.items():

        if amount <= 0:
            continue

        class_df = student_df[
            student_df["label_id"]
            == class_id
        ]

        sampled = rng.choice(
            class_df.index.to_numpy(),
            size=amount,
            replace=False,
        )

        selected_indices.extend(
            sampled.tolist()
        )

    if len(selected_indices) != n_samples:

        return None

    rng.shuffle(
        selected_indices
    )

    return student_df.loc[
        selected_indices
    ].copy()


def sample_h6_episode(
    student_df,
    k_shot,
    query_size,
    rng,
):
    """
    Construct a CLASS-BALANCED meta-training episode.

    Unlike test-time support sampling, this deliberately
    balances the support and query sets.

    Support:
        balanced K examples.

    Query:
        balanced query subset.

    We NEVER invent samples.

    For students missing classes, we balance over the
    classes that actually exist.
    """

    total_needed = (
        k_shot
        + query_size
    )

    if len(student_df) < total_needed:

        return (
            None,
            None,
            f"insufficient_clips_for_h6_"
            f"n={len(student_df)}_"
            f"needed={total_needed}",
        )

    # --------------------------------------------------------
    # IMPORTANT:
    # Select the full episode in a class-balanced manner
    # first, then split it into support/query.
    #
    # This prevents support/query from accidentally
    # competing for the same minority examples.
    # --------------------------------------------------------

    episode_df = sample_balanced_subset(
        student_df,
        total_needed,
        rng,
    )

    if episode_df is None:

        return (
            None,
            None,
            "h6_balanced_sampling_failed",
        )

    # Shuffle before splitting.
    episode_indices = (
        episode_df.index
        .to_numpy()
    )

    rng.shuffle(
        episode_indices
    )

    support_indices = (
        episode_indices[:k_shot]
    )

    query_indices = (
        episode_indices[k_shot:]
    )

    support_df = student_df.loc[
        support_indices
    ].copy()

    query_df = student_df.loc[
        query_indices
    ].copy()

    if (
        support_df.empty
        or query_df.empty
    ):

        return (
            None,
            None,
            "h6_empty_support_or_query",
        )

    return (
        support_df,
        query_df,
        None,
    )


# ============================================================
# CONTEXT ENCODER
# ============================================================

class ContextEncoder(nn.Module):

    def __init__(
        self,
        input_dim=INPUT_DIM,
        context_dim=CONTEXT_DIM,
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                256,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                256
            ),

            nn.Linear(
                256,
                context_dim,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                context_dim
            ),
        )

    def forward(
        self,
        support_x,
    ):

        encoded = self.net(
            support_x
        )

        context = encoded.mean(
            dim=0
        )

        return context


# ============================================================
# HYPERNETWORK
# ============================================================

class HyperNetwork(nn.Module):

    def __init__(
        self,
        context_dim=CONTEXT_DIM,
        hidden_dim=HYPER_HIDDEN,
        embed_dim=EMBED_DIM,
    ):

        super().__init__()

        self.embed_dim = embed_dim

        self.net = nn.Sequential(

            nn.Linear(
                context_dim,
                hidden_dim,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),

            nn.ReLU(),
        )

        self.output = nn.Linear(
            hidden_dim,
            HEAD_PARAM_DIM,
        )

    def forward(
        self,
        context,
    ):

        params = self.output(
            self.net(context)
        )

        weight_end = (
            NUM_CLASSES
            * self.embed_dim
        )

        weight = params[
            :weight_end
        ].view(
            NUM_CLASSES,
            self.embed_dim,
        )

        bias = params[
            weight_end:
        ]

        return (
            weight,
            bias,
        )


# ============================================================
# QUERY ENCODER
# ============================================================

class QueryEncoder(nn.Module):

    def __init__(
        self,
        input_dim=INPUT_DIM,
        embed_dim=EMBED_DIM,
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                256,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                256
            ),

            nn.Linear(
                256,
                embed_dim,
            ),

            nn.ReLU(),
        )

    def forward(
        self,
        x,
    ):

        return self.net(x)


# ============================================================
# FULL MODEL
# ============================================================

class HyperNetworkModel(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.context_encoder = (
            ContextEncoder()
        )

        self.query_encoder = (
            QueryEncoder()
        )

        self.hypernetwork = (
            HyperNetwork()
        )

    def generate_student_head(
        self,
        support_x,
    ):

        context = (
            self.context_encoder(
                support_x
            )
        )

        weight, bias = (
            self.hypernetwork(
                context
            )
        )

        return (
            weight,
            bias,
        )

    def predict(
        self,
        support_x,
        query_x,
    ):

        weight, bias = (
            self.generate_student_head(
                support_x
            )
        )

        query_emb = (
            self.query_encoder(
                query_x
            )
        )

        logits = F.linear(
            query_emb,
            weight,
            bias,
        )

        return logits


# ============================================================
# STANDARD CROSS ENTROPY
# ============================================================

def classification_loss(
    logits,
    targets,
):

    return F.cross_entropy(
        logits,
        targets,
    )


# ============================================================
# H6 META-TRAINING
# ============================================================

def episodic_train_h6(
    model,
    train_df,
    scaler,
    episodes,
    seed,
    device,
):
    """
    H6 meta-training.

    MAIN DIFFERENCE FROM ORIGINAL:

        Original:
            random K clips
            ↓
            usually dominated by Engage

        H6:
            class-balanced episode
            ↓
            minority classes appear much more often
            whenever the student actually has them

    Test-time protocol is NOT changed.
    """

    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    rng = np.random.default_rng(
        seed
    )

    groups = sorted(
        train_df["group"].unique()
    )

    if len(groups) == 0:

        raise ValueError(
            "No training students available."
        )

    successful_episodes = 0
    skipped_episodes = 0

    # --------------------------------------------------------
    # H6 episode sizes
    # --------------------------------------------------------

    support_options = [
        5,
        10,
        20,
    ]

    query_size = 64

    for episode in range(
        episodes
    ):

        # ----------------------------------------------------
        # Sample STUDENT/TASK
        # ----------------------------------------------------

        group = rng.choice(
            groups
        )

        student_df = train_df[
            train_df["group"]
            == group
        ].copy()

        k = int(
            rng.choice(
                support_options
            )
        )

        # ----------------------------------------------------
        # Construct BALANCED episode
        # ----------------------------------------------------

        (
            support_df,
            query_df,
            reason,
        ) = sample_h6_episode(
            student_df=student_df,
            k_shot=k,
            query_size=query_size,
            rng=rng,
        )

        if reason is not None:

            skipped_episodes += 1
            continue

        # ----------------------------------------------------
        # Transform features
        # ----------------------------------------------------

        support_x = scaler.transform(
            support_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        query_x = scaler.transform(
            query_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        query_y = query_df[
            "label_id"
        ].to_numpy(
            dtype=np.int64
        )

        support_x = torch.tensor(
            support_x,
            dtype=torch.float32,
            device=device,
        )

        query_x = torch.tensor(
            query_x,
            dtype=torch.float32,
            device=device,
        )

        query_y = torch.tensor(
            query_y,
            dtype=torch.long,
            device=device,
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        logits = model.predict(
            support_x,
            query_x,
        )

        # ----------------------------------------------------
        # H6 uses standard CE.
        #
        # The balancing happens at EPISODE CONSTRUCTION
        # rather than through aggressive class weighting.
        # ----------------------------------------------------

        loss = classification_loss(
            logits,
            query_y,
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        successful_episodes += 1

        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------

        if (
            (episode + 1) % 100
            == 0
        ):

            print(
                f"    Episode "
                f"{episode + 1}/{episodes} "
                f"loss={loss.item():.4f} "
                f"successful="
                f"{successful_episodes} "
                f"skipped="
                f"{skipped_episodes}"
            )

    print()
    print(
        "H6 meta-training complete:"
    )

    print(
        f"  Successful episodes: "
        f"{successful_episodes}"
    )

    print(
        f"  Skipped episodes: "
        f"{skipped_episodes}"
    )

    return model


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
):

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        average="macro",
        zero_division=0,
    )

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
    )

    return {
        "accuracy": float(
            accuracy
        ),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
        "confusion_matrix": cm,
    }


# ============================================================
# MAJORITY BASELINE
# ============================================================

def majority_prediction(
    y_query,
):

    counts = Counter(
        y_query.tolist()
    )

    majority_class = max(
        counts,
        key=counts.get,
    )

    return np.full(
        len(y_query),
        majority_class,
        dtype=np.int64,
    )


# ============================================================
# EVALUATION
# ============================================================

def evaluate_student(
    model,
    support_df,
    query_df,
    scaler,
    device,
):

    support_x = scaler.transform(
        support_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    query_x = scaler.transform(
        query_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    y_query = query_df[
        "label_id"
    ].to_numpy(
        dtype=np.int64
    )

    support_x = torch.tensor(
        support_x,
        dtype=torch.float32,
        device=device,
    )

    query_x = torch.tensor(
        query_x,
        dtype=torch.float32,
        device=device,
    )

    model.eval()

    with torch.no_grad():

        logits = model.predict(
            support_x,
            query_x,
        )

        y_pred = (
            torch.argmax(
                logits,
                dim=1,
            )
            .cpu()
            .numpy()
        )

    metrics = calculate_metrics(
        y_query,
        y_pred,
    )

    majority_pred = (
        majority_prediction(
            y_query
        )
    )

    majority_metrics = (
        calculate_metrics(
            y_query,
            majority_pred,
        )
    )

    return (
        y_query,
        y_pred,
        metrics,
        majority_metrics,
    )


# ============================================================
# RUN EXPERIMENT
# ============================================================

def run_experiment(
    clip_df,
    group_fold,
    k_values,
    seed,
    episodes,
    device,
    output_dir,
):

    eligible_groups = set(
        group_fold.keys()
    )

    eval_df = clip_df[
        clip_df["group"].isin(
            eligible_groups
        )
    ].copy()

    summary_rows = []
    per_student_rows = []
    per_class_rows = []
    confusion_rows = []
    skipped_rows = []

    fold_ids = sorted(
        set(group_fold.values())
    )

    # ========================================================
    # OUTER CROSS-VALIDATION
    # ========================================================

    for fold_id in fold_ids:

        print()
        print("=" * 80)
        print(
            f"H6 OUTER FOLD {fold_id}"
        )
        print("=" * 80)

        test_groups = [
            g
            for g, f
            in group_fold.items()
            if f == fold_id
        ]

        train_groups = [
            g
            for g, f
            in group_fold.items()
            if f != fold_id
        ]

        train_df = eval_df[
            eval_df["group"].isin(
                train_groups
            )
        ].copy()

        test_df = eval_df[
            eval_df["group"].isin(
                test_groups
            )
        ].copy()

        if train_df.empty:

            raise ValueError(
                f"No training clips "
                f"for fold {fold_id}"
            )

        # ----------------------------------------------------
        # Leakage check
        # ----------------------------------------------------

        overlap = (
            set(train_df["group"])
            &
            set(test_df["group"])
        )

        if overlap:

            raise RuntimeError(
                f"DATA LEAKAGE in fold "
                f"{fold_id}: {overlap}"
            )

        print(
            f"Training students: "
            f"{train_df['group'].nunique()}"
        )

        print(
            f"Test students: "
            f"{test_df['group'].nunique()}"
        )

        # ----------------------------------------------------
        # Training class distribution
        # ----------------------------------------------------

        print()
        print(
            "Training class counts:"
        )

        print(
            train_df[
                "label"
            ].value_counts()
        )

        # ----------------------------------------------------
        # Fit scaler ONLY on training data
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_train = train_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )

        scaler.fit(
            X_train
        )

        # ----------------------------------------------------
        # Fresh model
        # ----------------------------------------------------

        set_seed(
            seed + fold_id
        )

        model = HyperNetworkModel()

        model.to(device)

        print()
        print(
            f"H6 meta-training for "
            f"{episodes} episodes..."
        )

        episodic_train_h6(
            model=model,
            train_df=train_df,
            scaler=scaler,
            episodes=episodes,
            seed=(
                seed
                + fold_id * 1000
            ),
            device=device,
        )

        # ----------------------------------------------------
        # Test students
        # ----------------------------------------------------

        for group in test_groups:

            student_df = test_df[
                test_df["group"]
                == group
            ].copy()

            if student_df.empty:
                continue

            for k in k_values:

                for protocol in [
                    "random",
                    "early",
                ]:

                    local_seed = (
                        seed
                        + fold_id * 10000
                        + k * 100
                        + sum(
                            ord(ch)
                            for ch in group
                        )
                        + (
                            1
                            if protocol
                            == "early"
                            else 0
                        )
                    )

                    rng = (
                        np.random.default_rng(
                            local_seed
                        )
                    )

                    # ------------------------------------------------
                    # CRITICAL:
                    # Test support is sampled exactly as before.
                    # NO class balancing here.
                    # ------------------------------------------------

                    (
                        support_df,
                        query_df,
                        reason,
                    ) = sample_support_query(
                        student_df,
                        k,
                        protocol,
                        rng,
                    )

                    if reason is not None:

                        skipped_rows.append(
                            {
                                "outer_fold":
                                    fold_id,
                                "student_group":
                                    group,
                                "k_shot":
                                    k,
                                "support_protocol":
                                    protocol,
                                "reason":
                                    reason,
                            }
                        )

                        continue

                    (
                        y_true,
                        y_pred,
                        metrics,
                        majority_metrics,
                    ) = evaluate_student(
                        model=model,
                        support_df=support_df,
                        query_df=query_df,
                        scaler=scaler,
                        device=device,
                    )

                    # ------------------------------------------------
                    # Student-level row
                    # ------------------------------------------------

                    row = {

                        "method":
                            "hypernetwork_h6",

                        "outer_fold":
                            fold_id,

                        "student_group":
                            group,

                        "k_shot":
                            k,

                        "support_protocol":
                            protocol,

                        "support_size":
                            len(
                                support_df
                            ),

                        "query_size":
                            len(
                                query_df
                            ),

                        "accuracy":
                            metrics[
                                "accuracy"
                            ],

                        "balanced_accuracy":
                            metrics[
                                "balanced_accuracy"
                            ],

                        "macro_f1":
                            metrics[
                                "macro_f1"
                            ],

                        "majority_accuracy":
                            majority_metrics[
                                "accuracy"
                            ],

                        "majority_balanced_accuracy":
                            majority_metrics[
                                "balanced_accuracy"
                            ],

                        "majority_macro_f1":
                            majority_metrics[
                                "macro_f1"
                            ],
                    }

                    per_student_rows.append(
                        row
                    )

                    # ------------------------------------------------
                    # Per-class
                    # ------------------------------------------------

                    for class_id, label in enumerate(
                        LABELS
                    ):

                        per_class_rows.append(
                            {
                                "method":
                                    "hypernetwork_h6",

                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "class":
                                    label,

                                "precision":
                                    float(
                                        metrics[
                                            "precision"
                                        ][
                                            class_id
                                        ]
                                    ),

                                "recall":
                                    float(
                                        metrics[
                                            "recall"
                                        ][
                                            class_id
                                        ]
                                    ),

                                "f1":
                                    float(
                                        metrics[
                                            "f1"
                                        ][
                                            class_id
                                        ]
                                    ),

                                "support":
                                    int(
                                        metrics[
                                            "support"
                                        ][
                                            class_id
                                        ]
                                    ),
                            }
                        )

                    # ------------------------------------------------
                    # Confusion matrix
                    # ------------------------------------------------

                    cm = metrics[
                        "confusion_matrix"
                    ]

                    for true_id in range(
                        NUM_CLASSES
                    ):

                        for pred_id in range(
                            NUM_CLASSES
                        ):

                            confusion_rows.append(
                                {
                                    "method":
                                        "hypernetwork_h6",

                                    "outer_fold":
                                        fold_id,

                                    "student_group":
                                        group,

                                    "k_shot":
                                        k,

                                    "support_protocol":
                                        protocol,

                                    "true_class":
                                        LABELS[
                                            true_id
                                        ],

                                    "predicted_class":
                                        LABELS[
                                            pred_id
                                        ],

                                    "count":
                                        int(
                                            cm[
                                                true_id,
                                                pred_id
                                            ]
                                        ),
                                }
                            )

    # ========================================================
    # DATAFRAMES
    # ========================================================

    per_student_df = pd.DataFrame(
        per_student_rows
    )

    per_class_df = pd.DataFrame(
        per_class_rows
    )

    confusion_df = pd.DataFrame(
        confusion_rows
    )

    skipped_df = pd.DataFrame(
        skipped_rows
    )

    if per_student_df.empty:

        raise RuntimeError(
            "No student evaluations "
            "were produced."
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    summary_df = (
        per_student_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
            ],
            as_index=False,
        )
        .agg(

            n_students=(
                "student_group",
                "nunique",
            ),

            total_query_clips=(
                "query_size",
                "sum",
            ),

            mean_accuracy=(
                "accuracy",
                "mean",
            ),

            sd_accuracy=(
                "accuracy",
                "std",
            ),

            mean_balanced_accuracy=(
                "balanced_accuracy",
                "mean",
            ),

            sd_balanced_accuracy=(
                "balanced_accuracy",
                "std",
            ),

            mean_macro_f1=(
                "macro_f1",
                "mean",
            ),

            sd_macro_f1=(
                "macro_f1",
                "std",
            ),

            mean_majority_accuracy=(
                "majority_accuracy",
                "mean",
            ),

            mean_majority_balanced_accuracy=(
                "majority_balanced_accuracy",
                "mean",
            ),

            mean_majority_macro_f1=(
                "majority_macro_f1",
                "mean",
            ),
        )
        .sort_values(
            [
                "support_protocol",
                "k_shot",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    # ========================================================
    # SAVE
    # ========================================================

    os.makedirs(
        output_dir,
        exist_ok=True,
    )

    per_student_path = os.path.join(
        output_dir,
        "h6_per_student.csv",
    )

    summary_path = os.path.join(
        output_dir,
        "h6_summary.csv",
    )

    per_class_path = os.path.join(
        output_dir,
        "h6_per_class.csv",
    )

    confusion_path = os.path.join(
        output_dir,
        "h6_confusion_matrix.csv",
    )

    skipped_path = os.path.join(
        output_dir,
        "h6_skipped.csv",
    )

    per_student_df.to_csv(
        per_student_path,
        index=False,
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    per_class_df.to_csv(
        per_class_path,
        index=False,
    )

    confusion_df.to_csv(
        confusion_path,
        index=False,
    )

    skipped_df.to_csv(
        skipped_path,
        index=False,
    )

    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print()
    print("=" * 80)
    print(
        "H6: HYPERNETWORK + "
        "CLASS-BALANCED EPISODIC "
        "META-TRAINING RESULTS"
    )
    print("=" * 80)

    print(
        summary_df.to_string(
            index=False
        )
    )

    # ========================================================
    # PER-CLASS SUMMARY
    # ========================================================

    print()
    print("=" * 80)
    print(
        "H6 PER-CLASS RESULTS"
    )
    print("=" * 80)

    class_summary = (
        per_class_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
                "class",
            ],
            as_index=False,
        )
        .agg(

            mean_precision=(
                "precision",
                "mean",
            ),

            mean_recall=(
                "recall",
                "mean",
            ),

            mean_f1=(
                "f1",
                "mean",
            ),

            total_support=(
                "support",
                "sum",
            ),
        )
    )

    print(
        class_summary.to_string(
            index=False
        )
    )

    # ========================================================
    # SKIPPED
    # ========================================================

    print()
    print("=" * 80)
    print(
        "H6 SKIPPED CASES"
    )
    print("=" * 80)

    if skipped_df.empty:

        print("None.")

    else:

        print(
            skipped_df[
                "reason"
            ].value_counts()
        )

    # ========================================================
    # SAVE PATHS
    # ========================================================

    print()
    print("Saved:")
    print(
        per_student_path
    )
    print(
        summary_path
    )
    print(
        per_class_path
    )
    print(
        confusion_path
    )
    print(
        skipped_path
    )

    return (
        per_student_df,
        summary_df,
        per_class_df,
        confusion_df,
        skipped_df,
    )


# ============================================================
# MAIN
# ============================================================

def main():

    parser = argparse.ArgumentParser(
        description=(
            "H6 HyperNetwork meta-learning "
            "with class-balanced episodic "
            "meta-training."
        )
    )

    parser.add_argument(
        "--json",
        type=str,
        default=(
            r"/content/"
            r"final_data_1.json"
        ),
    )

    parser.add_argument(
        "--folds",
        type=str,
        default=(
            r"/content/"
            r"cmose_h1_h2_fold_protocol.csv"
        ),
    )

    parser.add_argument(
        "--k-values",
        type=str,
        default="5,10,20",
    )

    parser.add_argument(
        "--episodes",
        type=int,
        default=1000,
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=DEFAULT_SEED,
    )

    parser.add_argument(
        "--output-dir",
        type=str,
        default=(
            r"/content/"
            r"h6_results"
        ),
    )

    parser.add_argument(
        "--device",
        type=str,
        default="auto",
        choices=[
            "auto",
            "cpu",
            "cuda",
        ],
    )

    args = parser.parse_args()

    # ========================================================
    # DEVICE
    # ========================================================

    if args.device == "cuda":

        if not torch.cuda.is_available():

            raise RuntimeError(
                "CUDA requested but no "
                "CUDA device is available."
            )

        device = torch.device(
            "cuda"
        )

    elif args.device == "cpu":

        device = torch.device(
            "cpu"
        )

    else:

        device = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    print(
        f"Using device: {device}"
    )

    if device.type == "cuda":

        print(
            f"GPU: "
            f"{torch.cuda.get_device_name(0)}"
        )

    # ========================================================
    # SEED
    # ========================================================

    set_seed(
        args.seed
    )

    # ========================================================
    # K VALUES
    # ========================================================

    k_values = [
        int(v.strip())
        for v in args.k_values.split(",")
        if v.strip()
    ]

    if not k_values:

        raise ValueError(
            "No K values supplied."
        )

    # ========================================================
    # LOAD DATA
    # ========================================================

    print()
    print(
        "Loading clip data..."
    )

    clip_df = load_clip_dataframe(
        args.json
    )

    print(
        f"Loaded {len(clip_df)} "
        f"labelled clips from "
        f"{clip_df['group'].nunique()} "
        f"students."
    )

    print()
    print(
        "Overall class distribution:"
    )

    print(
        clip_df[
            "label"
        ].value_counts()
    )

    print()
    print(
        "Loading fold protocol..."
    )

    group_fold = (
        load_group_fold_map(
            args.folds
        )
    )

    print(
        f"Fold protocol contains "
        f"{len(group_fold)} students."
    )

    # ========================================================
    # RUN
    # ========================================================

    run_experiment(
        clip_df=clip_df,
        group_fold=group_fold,
        k_values=k_values,
        seed=args.seed,
        episodes=args.episodes,
        device=device,
        output_dir=args.output_dir,
    )


# ============================================================
# COLAB / NOTEBOOK SAFE EXECUTION
# ============================================================
#
# This avoids the:
#
#   unrecognized arguments: -f ...
#
# error caused by Jupyter/Colab.
#
# ============================================================

if __name__ == "__main__":

    import sys

    # In Colab/Jupyter, argparse sees the notebook's
    # "-f kernel.json" argument. We remove notebook
    # arguments and explicitly provide our own arguments.

    if "google.colab" in sys.modules:

        sys.argv = [
            "h6_hypernetwork.py",
            "--json",
            "/content/final_data_1.json",
            "--folds",
            "/content/cmose_h1_h2_fold_protocol.csv",
            "--k-values",
            "5,10,20",
            "--episodes",
            "1000",
            "--seed",
            "42",
            "--output-dir",
            "/content/h6_results",
            "--device",
            "cuda",
        ]

    main()

Using device: cuda
GPU: Tesla T4

Loading clip data...
Loaded 11902 labelled clips from 103 students.

Overall class distribution:
label
Engage              8245
Disengage           2170
Highly Engage       1145
Highly Disengage     342
Name: count, dtype: int64

Loading fold protocol...
Fold protocol contains 82 students.

H6 OUTER FOLD 1
Training students: 65
Test students: 17

Training class counts:
label
Engage              6291
Disengage           1601
Highly Engage        904
Highly Disengage     263
Name: count, dtype: int64

H6 meta-training for 1000 episodes...
    Episode 100/1000 loss=1.1838 successful=71 skipped=29
    Episode 400/1000 loss=0.6780 successful=295 skipped=105
    Episode 500/1000 loss=0.5045 successful=358 skipped=142
    Episode 600/1000 loss=0.8477 successful=439 skipped=161
    Episode 700/1000 loss=0.9677 successful=504 skipped=196
    Episode 800/1000 loss=0.4502 successful=580 skipped=220
    Episode 900/1000 loss=0.4695 successful=644 skipped=256

H6 m

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H6 OUTER FOLD 2
Training students: 65
Test students: 17

Training class counts:
label
Engage              6415
Disengage           1605
Highly Engage        893
Highly Disengage     259
Name: count, dtype: int64

H6 meta-training for 1000 episodes...
    Episode 200/1000 loss=1.4448 successful=146 skipped=54
    Episode 300/1000 loss=0.5347 successful=217 skipped=83
    Episode 500/1000 loss=0.8390 successful=360 skipped=140
    Episode 600/1000 loss=0.6361 successful=430 skipped=170
    Episode 700/1000 loss=0.7368 successful=496 skipped=204
    Episode 800/1000 loss=0.8018 successful=562 skipped=238
    Episode 900/1000 loss=0.4222 successful=635 skipped=265
    Episode 1000/1000 loss=0.3237 successful=710 skipped=290

H6 meta-training complete:
  Successful episodes: 710
  Skipped episodes: 290


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H6 OUTER FOLD 3
Training students: 66
Test students: 16

Training class counts:
label
Engage              6459
Disengage           1628
Highly Engage        872
Highly Disengage     251
Name: count, dtype: int64

H6 meta-training for 1000 episodes...
    Episode 200/1000 loss=0.6526 successful=153 skipped=47
    Episode 300/1000 loss=1.6692 successful=233 skipped=67
    Episode 400/1000 loss=1.1912 successful=310 skipped=90
    Episode 500/1000 loss=1.7790 successful=389 skipped=111
    Episode 800/1000 loss=0.8044 successful=632 skipped=168
    Episode 900/1000 loss=0.4527 successful=705 skipped=195
    Episode 1000/1000 loss=0.3980 successful=777 skipped=223

H6 meta-training complete:
  Successful episodes: 777
  Skipped episodes: 223


...


H6 OUTER FOLD 4
Training students: 66
Test students: 16

Training class counts:
label
Engage              6111
Disengage           1623
Highly Engage        815
Highly Disengage     237
Name: count, dtype: int64

H6 meta-training for 1000 episodes...
    Episode 100/1000 loss=1.1628 successful=68 skipped=32
    Episode 200/1000 loss=0.9680 successful=133 skipped=67
    Episode 300/1000 loss=1.0758 successful=202 skipped=98
    Episode 400/1000 loss=0.6418 successful=276 skipped=124
    Episode 500/1000 loss=0.5506 successful=350 skipped=150
    Episode 600/1000 loss=0.6668 successful=425 skipped=175
    Episode 700/1000 loss=0.5757 successful=493 skipped=207
    Episode 800/1000 loss=0.6210 successful=555 skipped=245
    Episode 1000/1000 loss=0.6047 successful=686 skipped=314

H6 meta-training complete:
  Successful episodes: 686
  Skipped episodes: 314


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H6 OUTER FOLD 5
Training students: 66
Test students: 16

Training class counts:
label
Engage              6568
Disengage           1727
Highly Engage        912
Highly Disengage     270
Name: count, dtype: int64

H6 meta-training for 1000 episodes...
    Episode 100/1000 loss=0.8813 successful=74 skipped=26
    Episode 200/1000 loss=0.6882 successful=148 skipped=52
    Episode 400/1000 loss=0.8146 successful=296 skipped=104
    Episode 500/1000 loss=0.6348 successful=367 skipped=133
    Episode 700/1000 loss=0.5900 successful=510 skipped=190
    Episode 800/1000 loss=0.7254 successful=577 skipped=223
    Episode 900/1000 loss=0.7655 successful=651 skipped=249
    Episode 1000/1000 loss=0.4713 successful=722 skipped=278

H6 meta-training complete:
  Successful episodes: 722
  Skipped episodes: 278


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H6: HYPERNETWORK + CLASS-BALANCED EPISODIC META-TRAINING RESULTS
support_protocol  k_shot  n_students  total_query_clips  mean_accuracy  sd_accuracy  mean_balanced_accuracy  sd_balanced_accuracy  mean_macro_f1  sd_macro_f1  mean_majority_accuracy  mean_majority_balanced_accuracy  mean_majority_macro_f1
           early       5          82              11016       0.593792     0.164842                0.463260              0.163823       0.330566     0.101807                0.708270                         0.307927                0.205566
           early      10          82              10606       0.595211     0.167093                0.461498              0.158376       0.325448     0.099766                0.710367                         0.315041                0.205928
           early      20          82               9786       0.593559     0.173209                0.462044              0.156551       0.321987     0.104130                0.713848                         0.325203   